# [M4] AI NHÌN VÀ HIỂU DỮ LIỆU
## Buổi S02: Tiền Xử Lý Ảnh & Kỹ Thuật Tăng Cường Dữ Liệu (Data Augmentation)

---

### Metadata Khóa Học
* **Module:** M4 — AI NHÌN VÀ HIỂU DỮ LIỆU
* **Buổi:** S02
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook thiết kế pipeline Data Augmentation bằng torchvision
* **Mức độ liên quan VOAI / Olympic AI:** Gia tăng đột biến độ chính xác tổng quát hóa khi thi đấu với tập dữ liệu ảnh nhỏ.

#### Mục tiêu học tập (Learning Objectives)
- Hiểu tại sao Data Augmentation là kỹ thuật bắt buộc để chống quá khớp trong các bài toán Thị giác máy tính có ít dữ liệu.
- Sử dụng thành thạo module `torchvision.transforms` để xây dựng chuỗi biến đổi hình ảnh.
- Làm chủ các phép biến đổi hình học phổ biến: RandomHorizontalFlip, RandomRotation, RandomResizedCrop.
- Hiểu và áp dụng chuẩn hóa ImageNet (Mean & Std Normalization) chuẩn mực cho các mô hình thị giác.

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M4_S01 (Biểu diễn ảnh số).
- Hiểu cơ chế hoạt động của PyTorch Dataset.

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Trong các cuộc thi Olympic AI, ban tổ chức thường chỉ cung cấp một tập dữ liệu ảnh rất khiêm tốn (vài trăm đến vài nghìn ảnh) vì giới hạn tài nguyên và thời gian chấm thi. Nếu bạn đưa nguyên những bức ảnh đó vào huấn luyện, mạng nơ-ron sâu sẽ nhanh chóng "ghi nhớ vẹt" từng vị trí góc cạnh của con mèo trong ảnh, dẫn đến Overfitting nghiêm trọng.

**Tăng cường dữ liệu (Data Augmentation)** là phương pháp "phù phép" giúp bạn nhân bản dữ liệu một cách thông minh mà không tốn công gán nhãn thêm: Một con mèo khi bị lật ngang, xoay nghiêng 10 độ, hay thay đổi độ sáng một chút thì nó VẪN LÀ CON MÈO. Augmentation dạy cho mạng nơ-ron học được các đặc trưng bất biến (Invariant Features) trước mọi sự biến đổi thực tế của môi trường.



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Bản chất của Data Augmentation trong Computer Vision
* **Mục tiêu:** Tạo ra các biến thể mới một cách ngẫu nhiên trong mỗi lần nạp batch ở pha huấn luyện (On-the-fly Transformation), làm phong phú phân phối dữ liệu mà không tốn dung lượng lưu trữ trên đĩa.
* **Nguyên tắc an toàn ngữ nghĩa (Semantic Safety):** Phép biến đổi phải bảo toàn nhãn của bài toán!
  - Lật ngang (`RandomHorizontalFlip`): An toàn cho phân loại chó/mèo, xe cộ, cây cối.
  - Lật dọc (`RandomVerticalFlip`): Nguy hiểm trong nhận diện chữ số (số 6 lật ngược biến thành số 9!).

### 2. Chuẩn hóa ImageNet Statistics
Hầu hết các mô hình thị giác máy tính đều được tiền chuẩn hóa dựa trên giá trị trung bình và độ lệch chuẩn của 1.4 triệu ảnh từ tập dữ liệu ImageNet:
* $\text{Mean} = [0.485, 0.456, 0.406]$
* $\text{Std} = [0.229, 0.224, 0.225]$
Công thức chuẩn hóa từng kênh:
$$x_{norm} = \frac{x - \text{mean}}{\text{std}}$$



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Quy trình Pipeline Tăng cường dữ liệu (Data Augmentation Pipeline):
[Ảnh gốc] ──> [Resize & Crop ngẫu nhiên] ──> [Lật ngang ngẫu nhiên 50%] ──> [Đổi màu ColorJitter] ──> [Chuẩn hóa Tensor]
     │
     └──> Mỗi Epoch, mạng nơ-ron lại nhìn thấy một diện mạo mới của cùng một bức ảnh!
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import torch
import torchvision.transforms as T
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# 1. Tao mot anh mau PIL Image gia lap co hinh chu nhat ro rang
np.random.seed(42)
img_array = np.zeros((100, 100, 3), dtype=np.uint8)
img_array[30:70, 20:50] = [255, 100, 50] # Hinh chu nhat mau cam o ben trai
sample_pil = Image.fromarray(img_array)

# 2. Thiet ke Pipeline Data Augmentation bang torchvision.transforms.Compose
train_transforms = T.Compose([
    T.Resize((128, 128)),
    T.RandomHorizontalFlip(p=0.5), # 50% xac suat lat ngang
    T.RandomRotation(degrees=20),   # Xoay ngau nhien trong khoang [-20, +20] do
    T.ColorJitter(brightness=0.3, contrast=0.3), # Bien thien do sang va tuong phan
    T.ToTensor(),                   # Chuyen PIL sang Tensor va chia cho 255.0
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Chuan hoa ImageNet
])

print("===== PIPELINE DATA AUGMENTATION CHO PHA TRAIN =====")
print(train_transforms)

### 🔍 Code Walkthrough (Khối 1)
Khối code xây dựng một pipeline tăng cường dữ liệu hoàn chỉnh bằng `torchvision.transforms.Compose`:
- Kết hợp liên hoàn các phép biến đổi hình học (Resize, Flip, Rotation) và quang học (ColorJitter).
- `ToTensor()` tự động đưa ảnh sang Tensor `[C, H, W]` và chuẩn hóa về $[0.0, 1.0]$.
- `Normalize()` đưa phân phối về chuẩn mực của các mô hình thị giác quốc tế.


In [ ]:
# 3. Kiem chung: Quan sat 4 bien the ngau nhien duoc sinh ra tu cung mot buc anh goc
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i in range(4):
    # Moi lan goi train_transforms se sinh ra mot bien the ngau nhien khac nhau
    augmented_tensor = train_transforms(sample_pil)
    
    # De hien thi duoc bang Matplotlib, can dao nguoc lai buoc Normalize (De-normalize)
    inv_normalize = T.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225]
    )
    img_display = inv_normalize(augmented_tensor).permute(1, 2, 0).clip(0, 1).numpy()
    
    axes[i].imshow(img_display)
    axes[i].set_title(f'Bien the ngau nhien #{i+1}', fontweight='bold')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

### 🔍 Code Walkthrough (Khối 2)
Khối code minh họa trực quan sự phong phú của dữ liệu sinh ra:
- Cùng một bức ảnh gốc, qua 4 lần gọi khác nhau đã tạo ra 4 diện mạo hoàn toàn mới (góc xoay khác nhau, vị trí lật trái/phải khác nhau, độ sáng khác nhau).
- Giúp mạng nơ-ron rèn luyện khả năng nhận diện hình dạng bất kể vị trí không gian.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Phân biệt Transform của Train vs Validation
Tại sao trong pha Validation và Test, chúng ta **tuyệt đối không được dùng `RandomHorizontalFlip` hay `RandomRotation`**, mà chỉ được dùng `Resize`, `CenterCrop` và `Normalize`?

### Bài 2 (Easy): Thử nghiệm phép làm mờ ảnh (GaussianBlur)
Thêm `T.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))` vào pipeline. Giải thích trong trường hợp camera thực tế bị rung hoặc mờ nét thì kỹ thuật này giúp ích gì.

### Bài 3 (Medium): Tăng cường dữ liệu bằng CutMix / MixUp
Tìm hiểu khái niệm kỹ thuật tiên tiến MixUp (trộn hai bức ảnh với nhau theo tỷ lệ $\lambda$ và $1-\lambda$). Giải thích tại sao MixUp lại cực kỳ phổ biến và mang lại điểm số cao trong các kỳ thi Kaggle và Olympic AI.

### Bài 4 (Medium): Đánh giá rủi ro ngữ nghĩa của Rotation
Nếu bài toán yêu cầu đọc biển báo giao thông (ví dụ biển cấm rẽ trái vs cấm rẽ phải), việc áp dụng `RandomHorizontalFlip(p=0.5)` có gây nguy hiểm không? Hãy giải thích.

### Bài 5 (Challenge): Viết Custom Transform Class
Viết một lớp `RandomAddNoise(noise_level=0.05)` kế thừa từ chuẩn của torchvision, tự động cộng thêm nhiễu Gauss ngẫu nhiên vào tensor ảnh: $x_{noisy} = x + 	ext{noise}$.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong các cuộc thi Olympic AI, sự khác biệt giữa thí sinh đạt giải Khuyến khích và thí sinh đạt Huy chương Vàng thường nằm ở **chiến lược Data Augmentation phù hợp với bài toán**:
Thí sinh thiếu kinh nghiệm sẽ áp dụng bừa bãi mọi kỹ thuật augmentation có sẵn trên mạng mà không phân tích tính chất ảnh (ví dụ lật ngược ảnh chụp X-quang tim phổi khiến quả tim bị đảo sang bên phải sai cấu trúc giải phẫu). Hiểu đúng ngữ nghĩa dữ liệu để chọn phép tăng cường hợp lý giúp bạn tối ưu hóa điểm số mà không làm hỏng bài thi.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Thí sinh áp dụng hàm `T.Normalize()` TRƯỚC KHI gọi `T.ToTensor()` và bị quẳng lỗi:

```python
# CODE CÓ LỖI THỨ TỰ TRANSFORMS:
# bad_pipe = T.Compose([
#     T.Normalize(mean=[0.5], std=[0.5]), # Loi: TypeError: img should be Tensor Image. Got PIL.Image!
#     T.ToTensor()
# ])
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao `T.Normalize()` bắt buộc phải đứng SAU `T.ToTensor()`.
2. Khẳng định quy tắc vàng về thứ tự transforms: Biến đổi trên PIL Image trước -> Chuyển sang Tensor -> Chuẩn hóa số học trên Tensor cuối cùng.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Kỹ thuật Tăng cường dữ liệu (Data Augmentation) nhằm giải quyết vấn đề lớn nhất nào trong Học sâu thị giác?
   - A. Giảm thời gian nạp dữ liệu từ đĩa.
   - B. Chống hiện tượng quá khớp (Overfitting) và tăng khả năng tổng quát hóa của mô hình khi tập dữ liệu huấn luyện có kích thước hạn chế.
   - C. Tự động tìm lỗi sai trong nhãn.
   - D. Tăng độ phân giải của máy ảnh.

2. Tại sao chúng ta không áp dụng các phép biến đổi ngẫu nhiên (Random Crop, Random Flip) cho tập Kiểm tra (Test set)?
   - A. Vì thư viện torchvision không hỗ trợ.
   - B. Vì tập Test cần đánh giá một cách khách quan, công bằng và có tính tất định (Deterministic) trên bức ảnh nguyên bản của người dùng.
   - C. Vì làm như vậy sẽ xóa mất tập Test.
   - D. Để tiết kiệm bộ nhớ RAM.

3. Trong bài toán nhận diện chữ số viết tay (MNIST), phép biến đổi nào sau đây có nguy cơ làm THAY ĐỔI BẢN CHẤT NHÃN của dữ liệu?
   - A. Xoay nhẹ 5 độ.
   - B. RandomVerticalFlip (Lật ngược theo chiều dọc biến số 6 thành số 9).
   - C. Thay đổi độ tương phản một chút.
   - D. Cắt viền trắng xung quanh.

4. Cặp giá trị thống kê `mean` và `std` chuẩn mực của ImageNet thường được áp dụng trên ảnh có thang đo nào?
   - A. Ảnh nguyên bản thang $[0, 255]$.
   - B. Ảnh Tensor đã được chuẩn hóa về thang số thực $[0.0, 1.0]$.
   - C. Ảnh xám nhị phân chỉ có 0 và 1.
   - D. Bất kỳ thang đo nào.

5. Lớp `torchvision.transforms.Compose` có chức năng chính là gì?
   - A. Huấn luyện mạng nơ-ron sâu.
   - B. Chuỗi hóa (Chain) nhiều phép biến đổi ảnh tuần tự lại với nhau thành một pipeline duy nhất.
   - C. Nén file ảnh sang định dạng zip.
   - D. Xuất dự đoán ra file CSV.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Cơ chế tăng cường dữ liệu ngẫu nhiên theo thời gian thực (On-the-fly Data Augmentation).
  - Cách dùng `torchvision.transforms.Compose` để xâu chuỗi các phép biến đổi.
  - Phân biệt rõ Transform dùng cho pha Train vs pha Validation.
  - Tầm quan trọng của chuẩn hóa ImageNet Statistics.
* **What I can now do:**
  - Thiết kế pipeline tiền xử lý và tăng cường ảnh chuyên nghiệp cho bài toán thị giác.
  - Bảo vệ mô hình khỏi hiện tượng học vẹt góc nhìn.
* **What I still need to practice:**
  - Khám phá kiến trúc chuyên dụng cho xử lý ảnh: **Mạng Tích Chập (CNN)** ở Buổi 3.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Trong **Buổi M4_S03**, chúng ta sẽ giải mã cấu trúc vĩ đại nhất của Computer Vision: **Trực Giác Mạng Tích Chập (CNN): Convolution, Stride, Padding & Pooling** — Tìm hiểu cơ chế trích xuất cạnh biên, toán tử tích chập ma trận 2D và cách giảm chiều không gian!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M4**:
- Data Augmentation trong Computer Vision ✓
- Sử dụng torchvision.transforms (Flip, Rotation, Jitter) ✓
- Chuẩn hóa ImageNet Statistics & Semantic Safety ✓

* **Session Deliverable:** Mã nguồn Pipeline Data Augmentation chuyên nghiệp cho ảnh số
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
